# Assignment 3.1: Multi-Objective Optimization with NSGA-II

## 🎯 Learning Objectives

In this assignment, you will:
- Understand multi-objective optimization concepts
- Learn about Pareto dominance and Pareto fronts
- Implement NSGA-II (Non-dominated Sorting Genetic Algorithm II)
- Work with ZDT benchmark problems
- Visualize Pareto fronts

This is a **production-ready implementation** of one of the most important MOO algorithms!

---

## 📦 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

# Import utilities
sys.path.append('../ga_toolkit')
from visualization import plot_pareto_front_2d

# Import NSGA-II implementation
from ga_utils_advanced import nsga2, get_zdt_problem

np.random.seed(42)

print("✅ Libraries imported successfully!")

---

## 🎓 2. Multi-Objective Optimization Basics

### 2.1 - What is Multi-Objective Optimization?

In many real-world problems, we need to optimize **multiple conflicting objectives** simultaneously.

**Examples:**
- Car design: Maximize performance AND minimize cost
- Portfolio: Maximize returns AND minimize risk
- Delivery: Minimize distance AND minimize time AND minimize vehicles

### 2.2 - Pareto Dominance

Solution **A dominates B** if:
- A is better or equal in ALL objectives
- A is strictly better in AT LEAST ONE objective

**Example:** (minimizing both objectives)
- A = (cost=10, time=5)
- B = (cost=12, time=7)
- A dominates B ✅

### 2.3 - Pareto Front

The **Pareto front** is the set of all non-dominated solutions (optimal trade-offs).

Let's implement dominance checking!

### Exercise 1: Implement Pareto Dominance Check

**Instructions:**
- `obj1` and `obj2` are arrays of objective values
- Return True if obj1 dominates obj2
- Assume minimization for all objectives

In [ ]:
def dominates(obj1, obj2):
    """
    Check if obj1 dominates obj2 (minimization).
    
    Arguments:
    obj1, obj2 -- numpy arrays of objective values
    
    Returns:
    True if obj1 dominates obj2
    """
    
    ### START CODE HERE ### (≈ 2-3 lines)
    # obj1 dominates if it's better/equal in all, and strictly better in at least one
    better_or_equal = np.all(obj1 <= obj2)
    strictly_better = np.any(obj1 < obj2)
    return better_or_equal and strictly_better
    ### END CODE HERE ###

In [ ]:
# Test your implementation
print("Testing dominance:")

test_cases = [
    (np.array([1, 2]), np.array([2, 3]), True),   # [1,2] dominates [2,3]
    (np.array([2, 3]), np.array([1, 2]), False),  # [2,3] does NOT dominate [1,2]
    (np.array([1, 3]), np.array([2, 2]), False),  # No dominance (trade-off)
    (np.array([1, 1]), np.array([1, 1]), False),  # Equal (no dominance)
]

for obj1, obj2, expected in test_cases:
    result = dominates(obj1, obj2)
    status = "✅" if result == expected else "❌"
    print(f"{status} {obj1} vs {obj2}: {result} (expected: {expected})")
    assert result == expected, f"Test failed for {obj1} vs {obj2}"

print("\n✅ All dominance tests passed!")

---

## 🔧 3. ZDT Benchmark Problems

ZDT (Zitzler-Deb-Thiele) are standard benchmark problems for multi-objective optimization.

### ZDT1 - Convex Pareto Front

$$f_1(x) = x_1$$
$$g(x) = 1 + \frac{9}{n-1}\sum_{i=2}^{n} x_i$$
$$f_2(x) = g(x) \cdot \left(1 - \sqrt{\frac{f_1(x)}{g(x)}}\right)$$

**Variables:** $x_i \in [0, 1]$, typically $n=30$

**Optimal Pareto front:** $f_2 = 1 - \sqrt{f_1}$ for $f_1 \in [0, 1]$

Let's load and test ZDT1:

In [ ]:
# Load ZDT1 problem
zdt1_funcs, zdt1_bounds, n_vars = get_zdt_problem('ZDT1')

print("ZDT1 Problem:")
print(f"  Number of variables: {n_vars}")
print(f"  Number of objectives: {len(zdt1_funcs)}")
print(f"  Bounds: {zdt1_bounds[0]} (all variables)")

# Test the functions
test_point = np.array([0.5] + [0]*29)  # x1=0.5, others=0
f1 = zdt1_funcs[0](test_point)
f2 = zdt1_funcs[1](test_point)

print(f"\nTest point: x1={test_point[0]}, x2...x30=0")
print(f"  f1 = {f1:.4f}")
print(f"  f2 = {f2:.4f}")
print(f"  This point is on the Pareto front: {np.isclose(f2, 1 - np.sqrt(f1), atol=0.01)}")

---

## 🚀 4. Run NSGA-II on ZDT1

Now let's run the complete NSGA-II algorithm!

In [ ]:
print("Running NSGA-II on ZDT1...")
print("="*70)

population, objectives, pareto_front, history = nsga2(
    objective_functions=zdt1_funcs,
    n_objectives=2,
    bounds=zdt1_bounds,
    pop_size=100,
    max_generations=100,
    crossover_rate=0.9,
    mutation_rate=1.0/n_vars,  # Recommended for ZDT
    eta_c=20,  # SBX parameter
    eta_m=20   # Polynomial mutation parameter
)

print("\n" + "="*70)
print("NSGA-II completed!")
print(f"  Final population size: {len(population)}")
print(f"  Pareto front size: {len(pareto_front)}")
print(f"  Generations: {len(history['n_pareto'])}")

---

## 📊 5. Visualize the Pareto Front

In [ ]:
# Plot Pareto front
plot_pareto_front_2d(
    objectives,
    pareto_front,
    title="NSGA-II on ZDT1",
    obj_labels=("f1", "f2")
)

# Add true Pareto front for comparison
true_pf_f1 = np.linspace(0, 1, 100)
true_pf_f2 = 1 - np.sqrt(true_pf_f1)
plt.plot(true_pf_f1, true_pf_f2, 'g--', linewidth=2, label='True Pareto Front', alpha=0.7)
plt.legend()
plt.show()

print("\n📊 The red points should align closely with the green dashed line!")

**Expected Result:**
- Red points (NSGA-II solutions) should closely follow the green dashed line (true Pareto front)
- The front should be well-distributed (thanks to crowding distance)
- Gray points are dominated solutions

---

## 📈 6. Analyze Convergence

In [ ]:
# Plot convergence metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pareto front size over generations
ax1.plot(history['n_pareto'], linewidth=2, color='blue')
ax1.set_xlabel('Generation', fontsize=12)
ax1.set_ylabel('Pareto Front Size', fontsize=12)
ax1.set_title('Pareto Front Growth', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Hypervolume (if available) or spread
if 'hypervolume' in history:
    ax2.plot(history['hypervolume'], linewidth=2, color='green')
    ax2.set_ylabel('Hypervolume', fontsize=12)
    ax2.set_title('Hypervolume Evolution', fontsize=14, fontweight='bold')
else:
    # Plot min/max of objectives as proxy for spread
    ax2.plot(range(len(history['n_pareto'])), linewidth=2, color='green')
    ax2.set_ylabel('Metric', fontsize=12)
    ax2.set_title('NSGA-II Progress', fontsize=14, fontweight='bold')

ax2.set_xlabel('Generation', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Pareto front size: {history['n_pareto'][-1]}")
print(f"Growth from gen 0: {history['n_pareto'][-1] - history['n_pareto'][0]}")

---

## 🎯 7. Compare Different ZDT Problems

Let's compare NSGA-II performance on different ZDT problems!

In [ ]:
# Quick comparison on ZDT2 (concave front)
print("Running NSGA-II on ZDT2 (concave front)...")

zdt2_funcs, zdt2_bounds, n_vars = get_zdt_problem('ZDT2')

pop2, obj2, pf2, hist2 = nsga2(
    objective_functions=zdt2_funcs,
    n_objectives=2,
    bounds=zdt2_bounds,
    pop_size=100,
    max_generations=100,
    mutation_rate=1.0/n_vars
)

print(f"\nZDT2 Pareto front size: {len(pf2)}")

In [ ]:
# Compare ZDT1 vs ZDT2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ZDT1 (convex)
ax1.scatter(objectives[:, 0], objectives[:, 1], alpha=0.3, s=20, color='gray', label='All')
ax1.scatter(objectives[pareto_front, 0], objectives[pareto_front, 1], 
           s=50, color='red', label='Pareto Front', zorder=5)
true_pf = 1 - np.sqrt(np.linspace(0, 1, 100))
ax1.plot(np.linspace(0, 1, 100), true_pf, 'g--', linewidth=2, label='True PF', alpha=0.7)
ax1.set_title('ZDT1 (Convex)', fontsize=12, fontweight='bold')
ax1.set_xlabel('f1')
ax1.set_ylabel('f2')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ZDT2 (concave)
ax2.scatter(obj2[:, 0], obj2[:, 1], alpha=0.3, s=20, color='gray', label='All')
ax2.scatter(obj2[pf2, 0], obj2[pf2, 1], 
           s=50, color='red', label='Pareto Front', zorder=5)
true_pf2 = 1 - np.linspace(0, 1, 100)**2
ax2.plot(np.linspace(0, 1, 100), true_pf2, 'g--', linewidth=2, label='True PF', alpha=0.7)
ax2.set_title('ZDT2 (Concave)', fontsize=12, fontweight='bold')
ax2.set_xlabel('f1')
ax2.set_ylabel('f2')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ NSGA-II successfully handles different Pareto front shapes!")

---

## 💡 8. Key Insights

### What We Learned:

1. **NSGA-II handles multiple objectives** - No need to manually weight objectives!

2. **Pareto fronts show trade-offs** - All solutions on the front are "optimal" in different ways

3. **Crowding distance maintains diversity** - Solutions spread evenly along the front

4. **Different problems, different fronts** - Convex (ZDT1) vs Concave (ZDT2)

### NSGA-II Components:

✅ **Non-dominated Sorting** - Ranks solutions by dominance  
✅ **Crowding Distance** - Maintains diversity  
✅ **SBX Crossover** - Simulated Binary Crossover for real values  
✅ **Polynomial Mutation** - Suitable for continuous variables  
✅ **Elitism** - Best solutions always survive  

---

## 🎯 9. Your Turn - Challenge Exercise!

**Challenge:** Try NSGA-II on ZDT3 (disconnected Pareto front)

ZDT3 has a **disconnected** Pareto front - multiple separate curves!

**Instructions:**
1. Load ZDT3 problem
2. Run NSGA-II
3. Plot the result
4. Observe the disconnected regions

In [ ]:
### YOUR CODE HERE ###
# Load ZDT3
zdt3_funcs, zdt3_bounds, n_vars = get_zdt_problem('ZDT3')

# Run NSGA-II (copy from above and modify)
pop3, obj3, pf3, hist3 = nsga2(
    objective_functions=zdt3_funcs,
    n_objectives=2,
    bounds=zdt3_bounds,
    pop_size=100,
    max_generations=150,  # May need more generations
    mutation_rate=1.0/n_vars
)

# Plot
plot_pareto_front_2d(obj3, pf3, title="ZDT3 - Disconnected Pareto Front")
plt.show()

print(f"\nZDT3 Pareto front points: {len(pf3)}")
print("You should see multiple disconnected curves! 🎯")

---

## 📚 10. Summary

### What You Accomplished:

✅ Understood multi-objective optimization  
✅ Implemented Pareto dominance checking  
✅ Ran NSGA-II on ZDT problems  
✅ Visualized Pareto fronts  
✅ Compared different problem types  
✅ Used a production-ready MOO algorithm  

### Real-World Applications:

- **Engineering Design:** Performance vs Cost vs Safety
- **Finance:** Return vs Risk vs Liquidity
- **Logistics:** Time vs Cost vs Quality (see Tutorial 5!)
- **Machine Learning:** Accuracy vs Speed vs Model Size

**Next:** Check out Tutorial 5 for a real industrial case study using these concepts!

---

## 🎉 Congratulations!

You've mastered multi-objective optimization with NSGA-II - one of the most important algorithms in evolutionary computation!

**Keep optimizing!** 🚀